# Run Phase G — Optuna tuning + Stacking (Colab)

This notebook installs dependencies, mounts Google Drive (optional), accepts uploads (or copies from Drive), moves files into ML-Model/, runs the `phase_g_optuna_stack.py` tuning script, and prints the generated report.

Notes: the notebook is robust to running from the Colab root (files uploaded to VM root) or from a Drive-mounted folder. Run cells top→bottom. Use GPU (T4) runtime.


In [ ]:
# 1) Install dependencies (may take a few minutes)
!pip install -q xgboost lightgbm scikit-learn tensorflow optuna joblib
print('Dependencies installed')


In [ ]:
# 2) (Optional) Mount Google Drive to persist models/logs
from google.colab import drive
print('If you want to persist outputs to Drive, run the next line and copy results after the run.')
# drive.mount('/content/drive')


In [ ]:
# 3) Upload required files (or skip if copied from Drive).
from google.colab import files
print('Upload phase_g_optuna_stack.py and balanced_water_potability_3000.csv (or a zip)')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))
!ls -la


In [ ]:
# 4) Ensure ML-Model folder exists and move uploaded files into it if present in root.
import os, shutil, glob
os.makedirs('ML-Model', exist_ok=True)
moved = []
candidates = ['phase_g_optuna_stack.py','balanced_water_potability_3000.csv']
for f in candidates:
    if os.path.exists(f):
        shutil.move(f, os.path.join('ML-Model', f))
        moved.append(f)
# If files already in a subfolder like ML-Model, leave them.
if os.path.exists('ML-Model/phase_g_optuna_stack.py') and os.path.exists('ML-Model/balanced_water_potability_3000.csv'):
    print('phase_g script and csv present in ML-Model/')
else:
    print('Files moved to ML-Model:', moved)
!ls -la ML-Model || true


In [ ]:
# 5) Run Phase G (Optuna tuning + stacking).
# This can take 30-90 minutes depending on trials and VM.
# Adjust --trials_xgb and --trials_lgb for shorter runs.
import sys
print('Running: python ML-Model/phase_g_optuna_stack.py --trials_xgb 80 --trials_lgb 60')
!python ML-Model/phase_g_optuna_stack.py --trials_xgb 80 --trials_lgb 60


In [ ]:
# 6) Show generated reports and latest report content
import glob, json
files = glob.glob('ML-Model/logs/*phase_g_report*.json')
if not files:
    files = glob.glob('logs/*phase_g_report*.json')
if files:
    latest = sorted(files)[-1]
    print('Latest report:', latest)
    print('---')
    print(open(latest).read())
else:
    print('No phase_g report found in ML-Model/logs or logs')


After completion: copy results to Drive if desired:
!cp -r ML-Model/logs /content/drive/MyDrive/ML-Model-PhaseG-Results/ || true
